# Variational quantum circuits for jet tagging (Chapters 4 and 5)

This notebook re-evaluates the trained variational quantum circuits (VQCs) of the thesis on small
demo sets of jets and compares them, jet by jet, with the stored outputs of the original training code.

**Chapter 4: the 1P1Q classifier** ("one particle, one qubit"). The 10 hardest constituents of a jet
sit on 10 qubits. Each particle is encoded by $R_Y(s\,p_T\,\Delta\eta)$ followed by
$R_X(s\,p_T\,\Delta\phi)$, a CNOT ring entangles the qubits, and one trainable $R_Y \to R_Z \to R_X$
block per qubit follows. The score is
$\langle Z_0\rangle + b$ (MSE loss, 32 parameters). Two models: $t\to bq\bar q$ vs. q/g (checkpoint
epoch 2) and $H\to c\bar c$ vs. q/g (epoch 9). These are the checkpoints with the best validation AUC,
and they are the ones the thesis evaluates.

**Chapter 5: re-uploading, readouts, pairwise entanglement.** The encode, entangle, rotate block is
repeated $L \in \{1, 2, 4, 8, 16\}$ times on the same particles (data re-uploading), with 7 or 10 qubits,
BCE or MSE loss, and one of three readouts:
$c_0\langle Z_0\rangle$, $\sum_i c_i\langle Z_i\rangle$, or
$\sum_i c_i\langle Z_i\rangle + \sum_{i<j} c_{ij}\langle Z_iZ_j\rangle$. The score is $f$ = readout + $b$
(a logit for BCE). The **$R_{YY}(\lambda_{ij}\Delta R_{ij})$ circuit** encodes $R_X(s\,p_T)$ only and
replaces the CNOT ring with an IsingYY gate on every particle pair, at angle $\lambda_{ij}\Delta R_{ij}$
($\lambda_{ij}$ is trainable per pair and shared by all layers). Datasets: JetClass ($t\to bq\bar q$ or
$W\to q\bar q$ vs. q/g jets from $Z(\nu\nu)$+jets) and JetsGame (top or W vs. QCD, 500 GeV, truth level).
Both datasets have the full 7-qubit grid: both circuits, all three readouts, BCE and MSE. Both also
have the 10-qubit BCE runs: 1P1Q with $\langle Z_0\rangle$ and $\sum_i c_i\langle Z_i\rangle$, and $R_{YY}$
with the pairwise readout. The summary table below lists every cell.

The encoding scale is $s = 2\pi\,\sigma(a) + 1$, with $a$ trainable. Inputs per particle: $\Delta\eta$ and
$\Delta\phi$ to the jet axis, mapped from $[-R, R]$ onto $[-\pi, \pi)$ ($R$ = 0.8 for JetClass, 1.0 for
JetsGame), and $p_T/p_{T,\mathrm{jet}}$. Parameter count: $3nL$ rotations, plus 2 ($a$, $b$), plus the
readout coefficients (1, $n$, or $n + n(n-1)/2$), plus $n(n-1)/2$ pair scales for $R_{YY}$.

**Requirements:** `pennylane` (tested with 0.42; `default.qubit`, numpy interface, so neither `jax` nor
`autograd` is needed for inference), `numpy`, `scikit-learn`; CPU only. Files: `thesis_models/vqc.py`
(circuits), `data/vqc_*.npz` (demo jets), `weights/vqc/` (weights, index, reference scores);
`tools/export_vqc.py` exports them from the training repositories and checks them.

**Demo sets: caveats.**
- Each set has 10,000 signal and 10,000 background jets, already scaled as the training readers scaled them.
  For Chapter 5 these are the first 10k jets of each class in the test files, out of the 50k + 50k test
  set every run was evaluated on. For Chapter 4 they are the first 10k of each class in the order of the
  thesis test loader.
- On 20k jets an AUC fluctuates by about ±0.002 around the full-test value, which is the number quoted in
  the thesis (100k jets for Chapter 5, 200k for Chapter 4).
- Every expectation value here is exact (analytic statevector). The Chapter 4 thesis scores were
  estimated with $10^4$ shots, so they differ from the exact values by shot noise (RMS ≈ 0.005).
  `shots=` reproduces that.
- Chapter 5 uses the final-epoch weights of one seed per grid cell (the first random seed in numerical order).

In [ ]:
import json, time
from collections import Counter
from pathlib import Path

import numpy as np
import pennylane as qml
from sklearn.metrics import roc_auc_score

from thesis_models.vqc import Ch4VQC, Ch5VQC, ch5_n_params, encoding_scale

DATA, WEIGHTS = Path("data"), Path("weights/vqc")

RUN_CH5_SELECTION = True        # representative Chapter 5 runs on all 20k jets each (~3 min on a CPU)
EXTRA_CH5_RUNS = {              # any other run of the index: name -> number of demo jets (None = all)
    "jetclass_e3_bce_q10_top_L1": 2000,
    "jetsgame_e3z0_bce_q7_wqcd_L16": 2000,
    "jetsgame_n_bce_q10_wqcd_L4": 2000,
}

DATASET = {"jetclass": "JetClass", "jetsgame": "JetsGame"}
TASK = {"top": "top vs q/g", "wqcd": "W vs q/g", "hcc": "H→cc̄ vs q/g"}
CIRCUIT = {"1P1Q": "1P1Q", "R_YY": "R_YY(λΔR)"}
READOUT = {"z0": "⟨Z0⟩", "z": "Σ⟨Zi⟩", "zz": "Σ⟨Zi⟩+Σ⟨ZiZj⟩"}
print(f"PennyLane {qml.__version__}, NumPy {np.__version__}")

In [ ]:
index = json.loads((WEIGHTS / "index.json").read_text())
ch4 = {r["name"]: r for r in index["ch4"]}
ch5 = {r["name"]: r for r in index["ch5"]}
w4, w5 = np.load(WEIGHTS / "ch4_weights.npz"), np.load(WEIGHTS / "ch5_weights.npz")
ref4 = np.load(WEIGHTS / "ch4_reference_scores.npz")   # thesis 10^4-shot scores, frozen-code analytic
ref5 = np.load(WEIGHTS / "ch5_reference_scores.npz")   # stored analytic test scores of each run
demo = {p.stem.removeprefix("vqc_"): dict(np.load(p)) for p in sorted(DATA.glob("vqc_ch*.npz"))}


def ch5_params(name):
    """Saved weights of one Chapter 5 run: rot (L, n, 3) and the auxiliary parameters."""
    return {k.split("/", 1)[1]: w5[k] for k in w5.files if k.startswith(name + "/")}


print(f"{'demo set':20s} {'signal vs background':26s} {'signal+background':>17s}   x (jets, particles, eta/phi/pT)   R")
for key, d in demo.items():
    y = d["y"]
    print(f"{key:20s} {str(d['signal']) + ' vs ' + str(d['background']):26s} "
          f"{int(y.sum()):>8d}+{int((y == 0).sum()):<8d}   {str(d['x'].shape):15s} {d['x'].dtype}"
          f"{'':11s}{float(d['jet_radius'])}")

print(f"\nChapter 4: {len(ch4)} models")
for r in ch4.values():
    print(f"  {r['name']:14s} {TASK[r['task']]:14s} 10 qubits, 1 layer, {r['n_params']} parameters, "
          f"checkpoint epoch {r['epoch']}, thesis test AUC {r['thesis_auc']:.4f} "
          f"({r['n_test_jets']} jets, {r['thesis_shots']} shots)")

print(f"\nChapter 5: {len(ch5)} runs (one seed per cell, final-epoch weights); "
      f"depths L = {sorted({r['depth'] for r in ch5.values()})}, tasks top and W vs q/g")
cols = [("BCE", 7), ("BCE", 10), ("MSE", 7)]
count = Counter((r["dataset"], r["circuit"], r["readout"], r["loss"], r["qubits"]) for r in ch5.values())
print(f"  {'dataset':9s} {'circuit':10s} {'readout':15s}" + "".join(f"{l} {q:>2d}q " for l, q in cols)
      + "   (runs = tasks x depths)")
for ds in DATASET:
    for c in CIRCUIT:
        for ro in READOUT:
            print(f"  {DATASET[ds]:9s} {CIRCUIT[c]:10s} {READOUT[ro]:15s}"
                  + "".join(f"{count[(ds, c, ro, l, q)] or '-':>7} " for l, q in cols))

In [ ]:
# Chapter 4: 10 qubits, 1 layer (gate structure; the angles depend on the jet and the weights).
m4 = Ch4VQC()
print(m4)
print(m4.draw(w4["ch4_top_ep02"], demo["ch4_top"]["x"][:1], decimals=None, max_length=130, show_matrices=False))
for name in ch4:
    print(f"{name}: encoding scale s = {encoding_scale(w4[name][-2]):.3f}, bias b = {w4[name][-1]:.4f}")

# Chapter 5, default circuit with re-uploading: 1P1Q <Z0>, 7 qubits, here L = 2 (gate structure only).
m = Ch5VQC("1P1Q", "z0", n_qubits=7, n_layers=2)
print("\n", m, sep="")
print(m.draw(ch5_params("jetclass_nz0_bce_q7_top_L2"), demo["ch5_jetclass_top"]["x"][:1],
             decimals=None, max_length=130, show_matrices=False))

# R_YY(lambda_ij dR_ij) with the pairwise readout, drawn for 4 qubits and 1 layer (the trained models
# have 7 or 10 qubits: 21 or 45 IsingYY gates per layer).
m = Ch5VQC("R_YY", "zz", n_qubits=4, n_layers=1)
print("\n", m, " (toy size, random parameters)", sep="")
print(m.draw(m.random_params(), demo["ch5_jetclass_top"]["x"][:1], decimals=None, max_length=130, show_matrices=False))

# Parameter counts: 3nL + 2 + readout coefficients (+ n(n-1)/2 pair scales for R_YY).
shapes = [(7, 1), (7, 16), (10, 1), (10, 16)]
print("\nparameters    " + "".join(f"  n={n:<2d} L={L:<3d}" for n, L in shapes))
for c in CIRCUIT:
    for ro in READOUT:
        print(f"{CIRCUIT[c]:10s} {READOUT[ro]:15s}"
              + "".join(f"{Ch5VQC(c, ro, n, L).n_params():>12d}" for n, L in shapes))

# Every exported weight set must have exactly the names/shapes the circuit expects.
for name, r in ch5.items():
    model = Ch5VQC(r["circuit"], r["readout"], r["qubits"], r["depth"])
    params = ch5_params(name)
    model.check_params(params)
    assert model.n_params() == ch5_n_params(r["circuit"], r["readout"], r["qubits"], r["depth"]) \
        == sum(np.size(v) for v in params.values()) == r["n_params"], name
for name in ch4:
    assert w4[name].shape == (m4.n_params(),)
example = "jetclass_e3_bce_q10_top_L16"
print(f"\n{example}: " + ", ".join(f"{k} {np.shape(v)}" for k, v in ch5_params(example).items())
      + f" -> {ch5[example]['n_params']} parameters")
print(f"all {len(ch5)} Chapter 5 weight sets and {len(ch4)} Chapter 4 weight vectors match their circuits")

In [ ]:
# Chapter 4: exact scores vs. the thesis' 10^4-shot scores of the same jets.
print(f"{'model':14s} {'task':14s} {'demo AUC':>9s} {'demo AUC':>9s} {'thesis AUC':>10s} "
      f"{'RMS(exact-':>10s} {'shot-noise':>10s} {'max|Δ| vs':>10s} {'time':>6s}")
print(f"{'':14s} {'':14s} {'exact':>9s} {'10^4 shots':>9s} {'200k jets':>10s} "
      f"{'10^4 shots)':>10s} {'RMS':>10s} {'frozen code':>10s}")
ch4_scores = {}
for name, r in ch4.items():
    d = demo[f"ch4_{r['task']}"]
    w = w4[name]
    t0 = time.time()
    f = m4.scores(w, d["x"])                            # <Z0> + b, analytic
    dt = time.time() - t0
    ch4_scores[name] = f
    shots, frozen = ref4[f"{name}/thesis_shots"], ref4[f"{name}/frozen_analytic"]
    z0 = f - w[-1]
    expected = np.sqrt(np.mean((1 - z0 ** 2) / r["thesis_shots"]))   # binomial noise of <Z0>
    print(f"{name:14s} {TASK[r['task']]:14s} {roc_auc_score(d['y'], f):9.4f} "
          f"{roc_auc_score(d['y'], shots):9.4f} {r['thesis_auc']:10.4f} "
          f"{np.sqrt(np.mean((f - shots) ** 2)):10.4f} {expected:10.4f} "
          f"{np.max(np.abs(f[:len(frozen)] - frozen)):10.1e} {dt:5.1f}s")

# The same estimate with finite shots: 10^4 shots on the first 1000 jets of the top model.
name, n = "ch4_top_ep02", 1000
f_shots = Ch4VQC(shots=10_000, seed=1).scores(w4[name], demo["ch4_top"]["x"][:n])
print(f"\n{name}, first {n} jets: RMS(10^4 shots here - exact) = "
      f"{np.sqrt(np.mean((f_shots - ch4_scores[name][:n]) ** 2)):.4f}, "
      f"RMS(thesis 10^4 shots - exact) = "
      f"{np.sqrt(np.mean((ref4[name + '/thesis_shots'][:n] - ch4_scores[name][:n]) ** 2)):.4f}")

In [ ]:
def run_ch5(name, n_jets=None):
    """Evaluate one Chapter 5 run on its demo set; compare with the run's stored test scores."""
    r = ch5[name]
    d = demo[f"ch5_{r['dataset']}_{r['task']}"]
    x, y = d["x"][:n_jets], d["y"][:n_jets]
    model = Ch5VQC(r["circuit"], r["readout"], r["qubits"], r["depth"])
    t0 = time.time()
    f = model.scores(ch5_params(name), x)               # readout + b, analytic
    ref = ref5[name][:len(f)]                           # stored scores (first 1000 jets for most runs)
    return dict(name=name, rec=r, n=len(f), auc=roc_auc_score(y, f),
                auc_stored=roc_auc_score(y, ref) if len(ref) == len(f) else np.nan,
                n_cmp=len(ref), max_diff=float(np.max(np.abs(f[:len(ref)] - ref))),
                seconds=time.time() - t0)


def print_ch5(rows):
    print(f"{'dataset':9s} {'task':11s} {'circuit':10s} {'readout':14s} {'loss':4s} {'n':>2s} {'L':>3s} "
          f"{'params':>6s} {'jets':>6s} {'demo AUC':>9s} {'(stored)':>9s} {'test AUC':>9s} "
          f"{'max|Δf|':>8s} {'time':>6s}")
    for row in rows:
        r = row["rec"]
        stored = f"{row['auc_stored']:9.4f}" if np.isfinite(row["auc_stored"]) else f"{'-':>9s}"
        print(f"{DATASET[r['dataset']]:9s} {TASK[r['task']]:11s} {CIRCUIT[r['circuit']]:10s} "
              f"{READOUT[r['readout']]:14s} {r['loss']:4s} {r['qubits']:2d} {r['depth']:3d} "
              f"{r['n_params']:6d} {row['n']:6d} {row['auc']:9.4f} {stored} "
              f"{r['test_auc']:9.4f} {row['max_diff']:8.1e} {row['seconds']:5.1f}s")


# Representative selection per dataset and task: default circuit 1P1Q <Z0> (BCE, 7 qubits) at
# L = 1 and 8, 1P1Q with the pairwise readout at L = 1, and R_YY with the pairwise readout at L = 2.
# "demo AUC" = this notebook on the 20k demo jets, "(stored)" = the stored scores of the same run on
# the same jets, "test AUC" = the run's stored AUC on its full 100k test set,
# max|Δf| = largest score difference to the stored scores on the same jets.
selection = [n for n, r in ch5.items() if r["notebook_default"]]
order = {"nz0_bce_q7_{}_L1": 0, "nz0_bce_q7_{}_L8": 1, "nzz_bce_q7_{}_L1": 2, "e3_bce_q7_{}_L2": 3}
selection.sort(key=lambda n: (ch5[n]["dataset"], ch5[n]["task"],
                              order[n.split("_", 1)[1].replace(ch5[n]["task"], "{}")]))
ch5_rows = [run_ch5(n) for n in selection] if RUN_CH5_SELECTION else []
print_ch5(ch5_rows)

In [ ]:
def find_runs(**filters):
    """Names of Chapter 5 runs matching index fields, e.g. find_runs(dataset="jetsgame", circuit="R_YY")."""
    return [n for n, r in ch5.items() if all(r[k] == v for k, v in filters.items())]


print("R_YY runs with <Z0> readout on JetsGame, BCE:", find_runs(dataset="jetsgame", circuit="R_YY",
                                                                  readout="z0", loss="BCE"))
# Any run of the grid: add its name to EXTRA_CH5_RUNS (setup cell). 10-qubit, deep circuits are slow
# on a CPU (about 1 min per 1000 jets for R_YY with 10 qubits and L = 16), so use a subset of jets.
print_ch5([run_ch5(n, n_jets) for n, n_jets in EXTRA_CH5_RUNS.items()])

In [ ]:
# Full check made with tools/export_vqc.py (verify): every Chapter 5 run of the index against its
# stored test scores on the same jets, and both Chapter 4 models against the thesis dumps.
ver = json.loads((WEIGHTS / "verification.json").read_text())
n_full = sum(v["n_jets"] == 20000 for v in ver["ch5"])
print(f"{len(ver['ch5'])} runs: {n_full} on all 20k demo jets, the others on the first "
      f"{min(v['n_jets'] for v in ver['ch5'])} jets")
groups = {}
for v in ver["ch5"]:
    r = ch5[v["name"]]
    groups.setdefault((CIRCUIT[r["circuit"]], r["qubits"]), []).append(v)
# max|Δf|: largest |notebook - stored| score difference over all jets of all runs in the group;
# |f32-f64 ΔR|: how much the scores move if ΔR_ij is formed in float64 instead of float32
# (first 200 jets per run), i.e. the float32 resolution of the R_YY entangling angles.
print(f"{'circuit':10s} {'n':>2s} {'runs':>5s} {'jets compared':>14s} {'max|Δf|':>9s} "
      f"{'median run':>11s} {'|f32-f64 ΔR|':>13s}")
for (c, q), vs in sorted(groups.items()):
    dr = [v["max_abs_f32_minus_f64_dr"] for v in vs if "max_abs_f32_minus_f64_dr" in v]
    print(f"{c:10s} {q:2d} {len(vs):5d} {sum(v['n_jets'] for v in vs):14d} "
          f"{max(v['max_abs_diff'] for v in vs):9.1e} {np.median([v['max_abs_diff'] for v in vs]):11.1e} "
          + (f"{max(dr):13.1e}" if dr else f"{'-':>13s}"))
full = [v for v in ver["ch5"] if "demo_auc" in v]
print(f"runs on all 20k jets: {len(full)}, max |demo AUC - AUC of stored scores| = "
      f"{max(abs(v['demo_auc'] - v['demo_auc_stored_scores']) for v in full):.1e}, "
      f"max |demo AUC - test AUC| = {max(abs(v['demo_auc'] - v['test_auc']) for v in full):.4f}")
for v in ver["ch4"]:
    print(f"{v['name']}: max|Δ| vs frozen 1P1Q code {v['max_abs_diff_frozen_analytic']:.1e}, "
          f"RMS vs thesis 10^4 shots {v['rms_vs_thesis_shots']:.4f} "
          f"(expected shot noise {v['expected_shot_rms']:.4f})")

**Reading the numbers.**
- **1P1Q circuits** (Chapters 4 and 5) reproduce the stored analytic scores of the training code to
  ~$10^{-14}$, i.e. bit for bit up to floating-point reassociation.
- **$R_{YY}(\lambda_{ij}\Delta R_{ij})$ circuits** agree to typically $10^{-7}$–$10^{-6}$, and to at most
  $2\times10^{-5}$ for the deepest BCE models. The training code formed $\Delta R_{ij}$ in single
  precision on a GPU, and the GPU rounds its last bit differently from NumPy. Forming $\Delta R_{ij}$ in
  double precision instead moves the scores by the same amount (column |f32-f64 ΔR|). The residual is
  therefore the float32 resolution of the entangling angles; it is not a difference in the circuit, and
  the AUCs agree to $<10^{-7}$.
- **Demo AUCs** scatter around the full-test AUCs within the statistics of a 20k-jet subset (up to
  ≈0.006). For Chapter 4 the demo jets give ≈0.005 more than the 200k-jet thesis value; the thesis' own
  $10^4$-shot scores of the same jets show the same offset, so it comes from the subset, not the circuit.
- **Chapter 4:** the RMS difference to the thesis scores equals the binomial shot noise of $10^4$ shots,
  $\sqrt{\langle(1-\langle Z_0\rangle^2)\rangle/10^4}$ ≈ 0.005.